# NextGen Boardroom

**Where bold ideas meet hard questions.**

## Pipeline

Runs **one detailed debate** for one company. The VC works through an agenda (market size first, to the CFO, then customers, unit economics, technology, go-to-market or risks); every main question is followed by a follow-up asking how the executive reached the answer. Asked for TAM, the CFO gives up to three sizing methods, each with its maths (checked in code) and assumptions.

Two more agents sit around the debate: a **Moderator** (JEV) who, after each VC question, says who will take it up ("This question will be taken up by the CFO."), and the **VC's Assistant** (fact-checker: JEV first, a stronger model only when unsure) who replies after each exchange with just "Facts are correct." or "Facts are incorrect.". The optimized run is compared with a naive run where Opus does everything.

Pipeline: **build corpus → ingest → live debate → (optional) cost baseline → evaluation dashboard → Markdown and HTML export**.

Every turn (with its maths, assumptions and citations), claim, fact-check, retrieval, model call, cost and the
VC decision is stored in `data/debate.sqlite`; checkpoints and VC memory live in Postgres. The last step writes
everything to Markdown in `reports/` and refreshes the local pages under `web/` from the saved SQLite transcript.

**Before you run:** kernel *Python (MultiAgentsForMarketSizing .venv)*; `.env` with `ANTHROPIC_API_KEY`
(and `JEV_API_KEY` for the JEV System-1 classifier, which then takes the first-pass fact-check verdicts); the licensed case PDF at
`case_study/biobot/01.BiobotAnalytics.pdf` and the public ledger `case_study/biobot/public_data.md`, plus the curated
`corpus/biobot/{facts,assumptions,sizing}.yaml`; Docker via Colima (the setup cell starts it).

**Cost:** live runs call Claude. Each run is capped at `limits.max_cost_usd_per_run` in `configs/models.yaml`($1 by default) and prints its measured cost; a 9-round debate is roughly $0.3–0.6 (estimate). The default is a live run. Set `DRY_RUN = True` only when you want scripted placeholder replies and no API spend.

## 1. Settings

In [21]:
COMPANY = "Biobot"
ROUNDS = 9                    # pitch + 4 agenda topics x (main question + follow-up); each round = question, answer, attack
PROFILE = "routed"            # "routed" = optimized: JEV for verdicts and the moderator, Haiku for claim extraction, Sonnet/Opus for the rest
DRY_RUN = False              # False = real agent conversation; True = scripted placeholders, no API cost
CLEAR_PREVIOUS_RUNS = True   # delete earlier debates (and reports) before running; ignored when RUN_DEBATE = False

# Turn the debate off to only (re)build the corpus and, if a finished live debate is already stored, evaluate and
# export it. Nothing is deleted and no model is called.
RUN_DEBATE = True             # False = skip the optimized debate (and the reset above) and reuse the latest stored one

# The naive baseline: the same debate with Opus on every call (moderator included). It runs once and the optimized debate
# is never repeated for it; the comparison table below needs both. Set False for exactly one (optimized) debate.
RUN_COST_BASELINE = True

EXPORT_DIR_NAME = "reports"   # Markdown output folder, relative to the project root

## 2. Setup: environment, Postgres, helpers

In [22]:
import contextlib, io, json, shutil, sqlite3, subprocess, sys, time

# A kernel that was started before the code last changed still holds the old `debate` modules. Drop them so this cell
# (and every cell after it) imports the current source; re-running it is enough, no kernel restart needed.
for _name in [m for m in sys.modules if m == "debate" or m.startswith("debate.")]:
    del sys.modules[_name]
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import Markdown, display

from debate.cli import DATA, main as debate_cli
from debate.config import ROOT, load_env

load_env()  # reads .env; values are never printed
import os
missing = [k for k in ("ANTHROPIC_API_KEY",) if not os.environ.get(k)]
if missing and not DRY_RUN:
    raise RuntimeError(f"set {missing} in .env (or use DRY_RUN = True)")
print("project:", ROOT)
print("System-1 verdicts:", f"JEV ({os.environ.get('JEV_MODEL', 'jev-latest')})" if os.environ.get("JEV_API_KEY") and PROFILE == "routed" and not DRY_RUN else "Claude Haiku (no JEV_API_KEY, dry run or opus_low)"); print("System-1 claim extraction: Claude Haiku (JEV only classifies)")

EXPORT_DIR = ROOT / EXPORT_DIR_NAME
EXPORT_DIR.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)


def sh(*cmd, check=True):
    return subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, check=check)


# Docker (Colima) + Postgres/pgvector: checkpoints, long-term memory and the RAG corpus live there.
if shutil.which("docker") is None:
    raise RuntimeError("docker CLI not found: brew install docker docker-compose colima")
if sh("docker", "info", check=False).returncode != 0:
    print("starting Colima ..."); sh("colima", "start", "--cpu", "2", "--memory", "2", "--disk", "10")
sh("docker", "compose", "up", "-d")
for _ in range(30):
    if "healthy" in sh("docker", "compose", "ps", "--format", "{{.Status}}", check=False).stdout:
        break
    time.sleep(2)
print(sh("docker", "compose", "ps", "--format", "{{.Name}}: {{.Status}}").stdout)

SESSION = {"started_at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "company": COMPANY,
           "rounds": ROUNDS, "profile": PROFILE, "dry_run": DRY_RUN, "runs": [], "reports": []}


def debate(*args, save_output_to=None):
    """Run a `debate` CLI command in this kernel; raise if it fails. Optionally save its printout to a file."""
    args = [str(a) for a in args]
    print("$ debate", " ".join(args))
    if save_output_to:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            code = debate_cli(args)
        print(buf.getvalue())
        Path(save_output_to).write_text(buf.getvalue())
    else:
        code = debate_cli(args)
    if code != 0:
        raise RuntimeError(f"`debate {args[0]}` exited with code {code}")


def _max_rowid():
    with sqlite3.connect(DATA / "debate.sqlite") as con:  # read-only: the store owns the schema
        if not con.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name='runs'").fetchone():
            return 0
        return con.execute("SELECT COALESCE(MAX(rowid), 0) FROM runs").fetchone()[0]


def run_debate(profile=PROFILE):
    """Run one debate and return its run id (also recorded in SESSION)."""
    before = _max_rowid()
    args = ["run", "--company", COMPANY, "--rounds", ROUNDS, "--profile", profile]
    args += ["--dry-run", "--with-rag"] if DRY_RUN else []
    code = debate_cli([str(a) for a in args])
    with sqlite3.connect(DATA / "debate.sqlite") as con:
        run_id, status, cost = con.execute(
            "SELECT run_id, status, total_cost_usd FROM runs WHERE rowid > ? ORDER BY rowid DESC LIMIT 1",
            (before,)).fetchone()
    SESSION["runs"].append({"run_id": run_id, "profile": profile, "status": status, "cost_usd": cost})
    if code != 0:
        raise RuntimeError(f"run {run_id} ended with status {status!r}; see the output above")
    return run_id

project: /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing
System-1 verdicts: JEV (jev-latest)
System-1 claim extraction: Claude Haiku (JEV only classifies)
multiagentsformarketsizing-postgres-1: Up 23 hours (healthy)



## 2b. Start clean (optional)

Deletes earlier debates so the database and `reports/` hold only this session's debate. The corpus is kept. Skipped when `RUN_DEBATE = False`.

In [23]:
if CLEAR_PREVIOUS_RUNS and RUN_DEBATE:
    debate("reset", "--yes", "--reports", EXPORT_DIR)
    EXPORT_DIR.mkdir(exist_ok=True)
else:
    print("keeping earlier debates (CLEAR_PREVIOUS_RUNS = False or RUN_DEBATE = False)")

$ debate reset --yes --reports /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports
deleted 2 files/folders, 2 checkpoint threads, 0 memory items


## 3. Build and ingest the corpus

`build-corpus` turns the case PDF into dated topic files, exhibit tables (Exhibit 7 reconciled against its own totals), dated endnotes and an index under `corpus/biobot/`, and turns the public ledger (`case_study/biobot/public_data.md`) into ~100 dated post-case items. It also validates the curated fact table (every quote checked against the PDF), the assumptions (ledger-sourced ones must resolve to exactly one dated item) and the sizing lineage. `ingest` embeds the ~360 chunks locally; `eval-retrieval` scores retrieval on 49 labelled questions. No model API calls.

In [24]:
debate("build-corpus", "--company", COMPANY)
debate("ingest", "--company", COMPANY)
debate("eval-retrieval", "--output", EXPORT_DIR / "retrieval_eval.md")
SESSION["reports"].append("retrieval_eval.md")
display(Markdown((ROOT / "corpus" / COMPANY / "README.md").read_text()))

$ debate build-corpus --company Biobot
built /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/case_study/corpus: 58 narrative chunks from 61 paragraphs, 3 exhibit tables, 5 captions, 44 dated endnotes, 98 dated public-ledger items
  exhibit check: Emergency Medical Services, FY16 expenditures: FY16 expenditures printed as '49,63,310' (malformed in the source); implied by the column total: 49,693,309
  exhibit check: FY16 expenditures: bureaus + standalone lines = 95,801,052 -> reconciles
  exhibit check: Total Public Health Service Centers FY16 expenditures: lines sum to 14,322,180, printed 14,302,181
  exhibit check: FY17 budget: bureaus + standalone lines = 98,022,291 -> reconciles
  exhibit check: FY18 budget: bureaus + standalone lines = 101,092,192 -> reconciles
index and sizing lineage: /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/case_study/corpus/README.md
$ debate ingest --company Biobot
ingested 364 dated chunks (21480 words) for Biobot: 7 assumption, 44 end

# Biobot corpus

Generated by `debate build-corpus` from HBS case 821-045, "Biobot Analytics". Every chunk is dated (`as_of`).
Generated folders are rebuilt from the PDF; `facts.yaml`, `assumptions.yaml` and `sizing.yaml` are curated by hand.

## Layout

| Path | What | Kind |
|---|---|---|
| `narrative/01_decision_context.md` | 3 paragraphs | generated |
| `narrative/02_founders_team.md` | 14 paragraphs | generated |
| `narrative/03_origin_research.md` | 5 paragraphs | generated |
| `narrative/04_science_and_prior_art.md` | 7 paragraphs | generated |
| `narrative/05_technology_ip.md` | 7 paragraphs | generated |
| `narrative/06_market.md` | 5 paragraphs | generated |
| `narrative/07_customers_go_to_market.md` | 7 paragraphs | generated |
| `narrative/08_competition_and_precedents.md` | 2 paragraphs | generated |
| `narrative/09_risks_and_objections.md` | 8 paragraphs | generated |
| `exhibits/ex5.csv` | Exhibit 5 as rows | generated |
| `exhibits/ex6.csv` | Exhibit 6 as rows | generated |
| `exhibits/ex7.csv` | Exhibit 7 as rows | generated |
| `captions.yaml` | 5 figure captions | generated |
| `sources/endnotes.yaml` | 44 endnotes, each with its publication date | generated |
| `sources/public.yaml` | Biobot public data and source ledger: one dated item per table row or bullet of `case_study/public_data.md` | generated |
| `facts.yaml` | atomic facts, quotes verified against the PDF | curated |
| `assumptions.yaml` | sizing inputs that are NOT in the case | curated |
| `sizing.yaml` | TAM methods recomputed from facts and assumptions | curated |

## Exhibit 7 reconciliation

- Emergency Medical Services, FY16 expenditures: FY16 expenditures printed as '49,63,310' (malformed in the source); implied by the column total: 49,693,309
- FY16 expenditures: bureaus + standalone lines = 95,801,052 -> reconciles
- Total Public Health Service Centers FY16 expenditures: lines sum to 14,322,180, printed 14,302,181
- FY17 budget: bureaus + standalone lines = 98,022,291 -> reconciles
- FY18 budget: bureaus + standalone lines = 101,092,192 -> reconciles

## Curated facts: 89

- claim: 5
- derived: 7
- estimate: 3
- forecast: 3
- plan: 2
- quote: 16
- reported: 53

## Public sources: 98 items

| Section | Items | Dated | Reliability |
|---|---:|---|---|
| Check of the requested milestones | 8 | 2017 to 2026-01-07 | sourced |
| History and operating milestones | 9 | 2017 to 2026 | sourced |
| Company and offering | 5 | 2025-11-24 to 2026-10-21 | sourced |
| Current product catalog and delivery model | 12 | 2026-10-22 | sourced |
| Competitors, alternatives, and procurement substitutes | 7 | 2026-10-14 to 2026-10-22 | sourced |
| Funding and investors | 5 | 2018-05 to 2025-06-18 | sourced |
| Contracts, customers, and adoption signals | 6 | 2021-05 to 2026-07 | sourced |
| Public measurements and reusable datasets | 8 | 2018 to 2026-10-22 | sourced |
| Named customer examples and what they show | 7 | 2018 to 2026-10-22 | sourced |
| Technology and intellectual property | 5 | 2019 to 2026-10-22 | sourced |
| Research and performance evidence | 5 | 2021 to 2026-10-22 | sourced |
| Business and market evidence, with limits | 6 | 2024-05-01 to 2026-10-22 | sourced |
| Research findings and measurement cautions | 3 | 2025 to 2026-10-22 | sourced |
| Screenshot-only funding leads supplied with this request | 12 | 2017-09-01 to 2026-10-22 | lead |

How each item was dated: latest year mentioned 24; latest cited publication date 17; year in first cell 2; event year in lead 6; manifest 1; section checked date 18; date column 19; ledger checked date (current-state item) 11.

Sizing inputs backed by public items: `people_per_catchment` <- `biobot_public_history_and_operating_03` (source); `price_per_sample` <- `biobot_public_company_and_offering_05` (cross_check); `municipal_price_floor` <- `biobot_public_company_and_offering_05` (source).

## Sizing lineage

### Beachhead budgets (top-down) = $511.39M

| Step | Input | Value | Source |
|---|---|---:|---|
|  | `beachhead_municipal_budgets` The four beachhead cities (Boston, San Francisco, New York City, Washi | $110.00B | fact (reported) |
| × | `public_health_share_estimate` The founders estimated 20% of those municipal budgets went to public h | 20.00% | fact (estimate) |
| × | `boston_recovery_share_fy18` Boston's Recovery Services Bureau was about 4.65% of its FY2018 Public | 4.65% | derived (derived) |
| × | `capture_beachhead` Share of the beachhead cities' recovery-services spend Biobot could wi | 50.00% | assumption (assumption) |

### Share of an existing market (top-down) = $744.23M

| Step | Input | Value | Source |
|---|---|---:|---|
|  | `water_market_2022_forecast` The water and wastewater treatment market was forecast (in a 2016 repo | $138.00B | fact (forecast) |
| × | `municipal_share_of_water_market` Municipal buyers were about 53.9% of the 2015 water treatment market. | 53.93% | derived (derived) |
| × | `capture_existing_market` Share of the municipal water-treatment market Biobot could capture. | 1.00% | assumption (assumption) |

### Manhole sampling (bottom-up) = $6.51B

| Step | Input | Value | Source |
|---|---|---:|---|
|  | `world_population` World population used for the bottom-up sizing. | 7,750,000,000 people | assumption (external_unverified) |
| × | `urban_share_claim` Ghaeli said 70% of the world's population lived in cities (her claim;  | 70.00% | fact (claim) |
| ÷ | `people_per_catchment` People served by one sampling site (catchment) in the bottom-up sizing | 10,000 people | public source (public_source) |
| × | `samples_per_year` Monthly sampling means 12 samples per site per year. | 12 samples per site per year | derived (derived) |
| × | `price_per_sample` Price Biobot charges per analysed sample in the bottom-up sizing. | $1,000 | assumption (assumption) |

### City contracts at the reported price floor (bottom-up) = $13.05M

> A floor, not a TAM: it prices one minimum contract per city. The public ledger warns the $15,000 floor cannot be multiplied by the cumulative number of tested sites to estimate revenue (many sites were pro-bono, federally funded or part of one contract), and it mixes a 2017 city count with a 2025 price.

| Step | Input | Value | Source |
|---|---|---:|---|
|  | `cities_including_smaller` Including smaller US and Canadian cities, the founders counted 870 pot | 870 cities | fact (estimate) |
| × | `municipal_price_floor` Minimum a municipality pays Biobot per year for its service, as report | $15,000 | public source (public_source) |



## 4. The debate

Set `RUN_DEBATE = False` to skip this step and reuse the latest finished live debate already stored. Each turn prints the point, the maths (re-checked in code) and the assumptions behind it; VC follow-ups are marked `[follow-up]`.

In [25]:
if RUN_DEBATE:
    RUN_ID = run_debate()
else:
    RUN_ID = None  # reuse the latest finished live debate in the store, if there is one (read-only lookup)
    with sqlite3.connect(DATA / "debate.sqlite") as con:
        if con.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name='runs'").fetchone():
            for run_id, status, cost, config in con.execute(
                    "SELECT run_id, status, total_cost_usd, config_json FROM runs WHERE status = 'completed' ORDER BY rowid DESC"):
                if not json.loads(config).get("dry_run"):
                    RUN_ID = run_id
                    SESSION["runs"].append({"run_id": run_id, "profile": PROFILE, "status": status, "cost_usd": cost})
                    break
    print(f"RUN_DEBATE = False: {'using stored run ' + RUN_ID if RUN_ID else 'no finished live debate stored; skipping the steps below'}")
RUN_ID


ImportError: cannot import name 'RouteChoice' from 'debate.state' (/Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/src/debate/state.py)

## 5. Naive baseline: the same debate with Opus on every call

Runs once, only when `RUN_COST_BASELINE = True`; the optimized debate above is **not** run again (with `RUN_DEBATE = False` it reuses the stored one). Every call (debate, claim extraction, fact-check verdicts, escalations and the moderator who picks who answers) goes to Opus at low effort. Costs are real, so the per-run cap in `configs/models.yaml` is $3.

In [ ]:
if RUN_COST_BASELINE and RUN_ID:
    if PROFILE != "routed":
        raise ValueError("the baseline compares against a routed run; set PROFILE = 'routed'")
    OPUS_RUN = run_debate(profile="opus_low")
    cost_report = EXPORT_DIR / "cost_comparison.md"
    debate("compare", "--routed", RUN_ID, "--baseline", OPUS_RUN, save_output_to=cost_report)
    cost_report.write_text("# Cost: routed vs Opus-only\n\n```\n" + cost_report.read_text() + "```\n")
    SESSION["reports"].append(cost_report.name)
else:
    print("skipped (RUN_COST_BASELINE = False, or no optimized debate to compare with)")

Biobot Analytics | run 5ccc7c373555



[1] VC -> CEO
    Let's start simple: in one sentence, what does Biobot do, for whom, and why now? Specifically, who is the first buyer you are building for, and what makes this the right moment to sell wastewater-based drug-consumption data rather than five years from now?
    (957 in + 2489 cache-write + 0 cache-read / 299 out, 5495 ms, $0.02225)



[2] CEO -> VC
    Biobot turns sewage into near-real-time drug-consumption data, starting with opioids, and our first buyers are state and municipal governments and public health departments. Why now: the underlying science is over ten years mature and public interest in opioids is very high. That said, procurement and budget friction with cities is a real risk I won't downplay.
    cites: biobot_ex6_prescription_illicit_drugs, biobot_fact_opioids_first
    (1215 in + 2169 cache-write + 764 cache-read / 433 out, 5900 ms, $0.02452)



[3] DEVIL -> CEO
    The 'why now' argument undercuts itself: science mature for over ten years (EPA hypothesis 2001, Oregon 2008, Oslo 2011) means academics and treatment facilities already do this, so maturity is no moat. Exhibit 5 lists health departments, EMS and police data as competition. My inference is that cities may see Biobot as redundant to data they already fund.
    cites: biobot_case_p4_c2, biobot_ex5_municipal_governments
    (1262 in + 2293 cache-write + 764 cache-read / 501 out, 6918 ms, $0.02669)



[4] VC -> the panel
    How big is the opportunity in dollars, and show the maths. Give me at least two independent sizing methods, for example a bottom-up count of paying cities times annual contract price, and a top-down share of public-health surveillance budgets. Which inputs come from data and which are your guesses, and why might the methods disagree?
    (1488 in + 1882 cache-write + 764 cache-read / 304 out, 5235 ms, $0.02159)


...


[31] DEVIL -> CFO [rewritten x1]
    The CFO's survival test rests on a mismatch. The $250 threshold applies a 75% cut to an assumed $1,000 sample price, but the Massachusetts discount comes from a $400,000/month statewide contract. At $1,000 per sample, that contract implies about 400 samples a month. My inference: the per-sample price is unanchored, so the break-even test is meaningless.
    cites: biobot_public_contracts_customers_and_06, biobot_assume_price_per_sample
    maths: $400,000/month / $1,000/sample = 400 samples/month
    assumes: Price of $1,000 per sample is a TAM-slide assumption, not a case fact; The contract value is entirely per-sample fees
    (4589 in + 0 cache-write + 5350 cache-read / 922 out, 14759 ms, $0.03787)



[32] VC -> the panel [follow-up]
    Your $1,000-per-sample price is a TAM-slide assumption, and the Devil showed it implies about 400 samples a month for Massachusetts' $400,000/month contract. Which input breaks your model, price or cost to serve? At a 75% undercut, what break-even sample volume per month keeps you viable, and where does that number come from?
    cites: biobot_assume_price_per_sample, biobot_public_contracts_customers_and_06
    (2575 in + 0 cache-write + 2646 cache-read / 442 out, 11516 ms, $0.01967)

[33] MODERATOR -> CFO
    This follow-up stays with the CFO, who gave the answer being probed.
    (0 in + 0 cache-write + 0 cache-read / 0 out, 0 ms, $0.00000)



[34] CFO -> VC [rewritten x1]
    Cost to serve breaks the model: it is unknown, so no break-even volume is defensible. My $1,000 price is a TAM-slide assumption; I concede the Devil's point that applying a 75% cut to it was unanchored. The only market anchor is July 2026 Massachusetts, where a $400,000/month contract drops to about $100,000/month in-house. Any break-even volume would be a guess.
    cites: biobot_assume_price_per_sample
    maths: $400,000/month x 25% = $100,000/month; $100,000/month / $1,000/sample = 100 samples/month
    assumes: Price of $1,000 per sample is a TAM-slide assumption, not a case fact; Contract value is entirely per-sample fees; Cost to serve per sample is unknown
    (4857 in + 0 cache-write + 5336 cache-read / 1071 out, 14301 ms, $0.04192)



[35] DEVIL -> CFO [rewritten x1]
    The CFO's 100 samples/month figure divides the in-house replacement cost by Biobot's assumed price, which mixes a public lab's budget with Biobot's revenue. That number measures nothing Biobot earns. Per the July 2026 Globe report, the $100,000/month is Massachusetts' own cost. My inference: Biobot keeps zero of it, so revenue at risk is the full $400,000/month.
    cites: biobot_public_contracts_customers_and_06, biobot_assume_price_per_sample
    maths: $400,000/month x 12 months = $4,800,000/year contract value lost
    assumes: Inference: Massachusetts' in-house program pays Biobot nothing
    (4914 in + 0 cache-write + 5350 cache-read / 950 out, 16033 ms, $0.03973)



VC decision: pass (conviction 80%)

claims: 2 contradicted, 115 not_checked, 66 supported, 11 unverifiable
model calls: 85 (0 System-1, 0%); 201729 in / 29281 out; $1.4964 billed from usage

completed: 35 turns, $1.4964 total including checks
stored in /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/data/debate.sqlite (run 5ccc7c373555); checkpoints and memory in Postgres
$ debate compare --routed 5318f7e0717c --baseline 5ccc7c373555
biobot | recorded usage
routed: run 5318f7e0717c, $0.5499, 82 calls, 44% System-1
opus_low: run 5ccc7c373555, $1.4964, 85 calls, 0% System-1
routed savings: $0.9465 (63.3% of baseline)



## 5b. Optimized vs naive: tokens, latency, cost

One table for the whole conversation of each run, written to `reports/optimized_vs_opus_only.md` (plus `reports/inference_comparison.md` and the complete stored results of every live run in `reports/raw/<run_id>.json`). Wall-clock time is start to finish of the debate; total call latency adds up the time of every model call.

In [ ]:
res = sh(sys.executable, str(ROOT / "scripts" / "inference_metrics.py"), "--out", str(EXPORT_DIR / "inference_comparison.md"),
         "--versus-out", str(EXPORT_DIR / "optimized_vs_opus_only.md"))
print(res.stdout.strip())
display(Markdown((EXPORT_DIR / "optimized_vs_opus_only.md").read_text()))
SESSION["reports"] += ["optimized_vs_opus_only.md", "inference_comparison.md"]

wrote /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/inference_comparison.md
wrote /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/optimized_vs_opus_only.md (measured baseline)
wrote reports/raw/5318f7e0717c.json
wrote reports/raw/5ccc7c373555.json


# Optimized (routed) vs Opus-only

Company biobot, 9 rounds. Both columns are measured runs.

| Metric | Optimized (routed) | Opus only |
|---|---:|---:|
| Models | JEV (fact-check verdicts, moderator) + Haiku (claim extraction) + Sonnet (debate, escalations) + Opus (decision) | Opus for every call, including the moderator |
| Run | `5318f7e0717c` (measured) | `5ccc7c373555` (measured) |
| Model calls | 82 | 85 |
| Input tokens | 226,689 | 201,729 |
| Output tokens | 31,706 | 29,281 |
| Cache-read tokens | 83,037 | 102,428 |
| Wall-clock time for the whole debate (s) | 225 | 419 |
| Total call latency, summed over calls (s) | 223 | 417 |
| Total tokens (input + output) | 258,395 | 231,010 |
| JEV calls | 13 | 0 |
| Moderator picks (CEO / CFO / CTO) | 1 / 2 / 1 | 1 / 2 / 1 |
| Cost (USD) | $0.5499 | $1.4964 |
| Cost per turn (USD) | $0.0157 | $0.0428 |
| Avg words per turn | 41.3 | 44.3 |
| Claims extracted (supported) | 205 (38) | 194 (66) |
| VC decision | pass | pass |
| Saving from routing | $0.9465 (63% of Opus-only) |  |
| Wall-clock difference (optimized vs Opus-only) | -194s (-46%) |  |


## 6. Evaluation dashboard

Reads saved records only; no model calls.

In [ ]:
if RUN_ID:
    debate("eval-report", "--company", COMPANY, "--output", EXPORT_DIR / "evaluation_dashboard.md")
    SESSION["reports"].append("evaluation_dashboard.md")
else:
    print("skipped: no debate to evaluate")


$ debate eval-report --company biobot --output /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/evaluation_dashboard.md
evaluation dashboard: /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports/evaluation_dashboard.md


## 7. Export the transcript and refresh the static pages

Write one Markdown report per run, then embed the optimized and Opus-only conversations and their selected VC decision memo in `web/biobot.html`. Local exports can contain case-derived dialogue; the GitHub Pages workflow replaces them with the public demo.

In [ ]:
session_ids = [r["run_id"] for r in SESSION["runs"]]
# Export only this session's debate (plus the baseline, if you switched it on).
if session_ids:
    debate("export", *session_ids, "--out", EXPORT_DIR)
    page_args = [sys.executable, str(ROOT / "scripts" / "build_report_page.py")]
    if DRY_RUN:
        page_args.append("--include-dry-runs")
    print(sh(*page_args).stdout.strip())
else:
    print("skipped: no debate to export")

SESSION["finished_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
SESSION["total_cost_usd"] = round(sum(r["cost_usd"] or 0 for r in SESSION["runs"]), 6)
stamp = SESSION["started_at"].replace(":", "").replace("-", "")[:15]
(EXPORT_DIR / f"session_{stamp}.json").write_text(json.dumps(SESSION, indent=2))
print(f"session cost: ${SESSION['total_cost_usd']:.4f}")
print(*sorted(str(p.relative_to(ROOT)) for p in EXPORT_DIR.rglob("*.md")), sep="\n")

$ debate export 5318f7e0717c 5ccc7c373555 --out /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports
wrote 3 Markdown files to /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/reports
Embedded 1 saved runs in /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/index.html, /Users/isha/Desktop/Projects/MultiAgentsForMarketSizing/conversation.html
session cost: $2.0463
reports/cost_comparison.md
reports/evaluation_dashboard.md
reports/index.md
reports/inference_comparison.md
reports/optimized_vs_opus_only.md
reports/retrieval_eval.md
reports/runs/5318f7e0717c.md
reports/runs/5ccc7c373555.md


## 8. Read the results here or open the HTML conversation

In [ ]:
if RUN_ID:
    display(Markdown((EXPORT_DIR / "index.md").read_text()))


---
title: "Debate runs"
---

# Debate runs

| Run | Company | Status | Decision | Turns | Cost | Started |
|---|---|---|---|---:|---:|---|
| [5318f7e0717c](runs/5318f7e0717c.md) | biobot | completed | pass | 35 | $0.5499 | 2026-10-22T16:56:02+00:00 |
| [5ccc7c373555](runs/5ccc7c373555.md) | biobot | completed | pass | 35 | $1.4964 | 2026-10-22T16:59:47+00:00 |

## Reports

- [cost comparison](cost_comparison.md)
- [evaluation dashboard](evaluation_dashboard.md)
- [inference comparison](inference_comparison.md)
- [optimized vs opus only](optimized_vs_opus_only.md)
- [retrieval eval](retrieval_eval.md)


In [ ]:
# Full transcript with maths, assumptions, fact-checks and the decision
if RUN_ID:
    display(Markdown((EXPORT_DIR / "runs" / f"{RUN_ID}.md").read_text().split("---", 2)[-1]))




# Biobot debate (5318f7e0717c)

## Summary

| | |
|---|---|
| Status | completed |
| Decision | pass |
| Turns | 35 |
| Claims | 116 not_checked, 48 unverifiable, 38 supported, 2 miscalculated, 1 contradicted |
| Model calls | 82 (36 System-1) |
| Tokens | 226,689 in / 31,706 out |
| Cost | $0.5499 |
| Prompt version | `28c37dcf` |

## Transcript

### 1. VC → CEO (round 1)

Thanks for joining. In one sentence, what does Biobot do, for whom, and why is now the right time to build it?


<sub>claude-sonnet-5-5 · 915 in / 166 out · 5720 ms · $0.00691</sub>

### 2. CEO → VC (round 1)

Biobot turns wastewater into near-real-time data on community drug consumption, starting with opioids for city governments that currently lack timely measures. Why now: the opioid crisis is pressing, and the 2021 Series A shows investor backing. I cannot verify from this evidence how many cities pay.

**Assumptions:**

- Cities currently lack timely local measures of opioid consumption

**Cites:** `biobot_fact_opioids_first` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_public_funding_and_investors_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: Biobot turns wastewater into near-real-time data on community drug consumption. — Evidence shows wastewater drug-consumption tracking, with real-time monitoring aims and alert lead times.
- ✅ _supported_: Biobot starts with opioids as its focus, serving city governments that currently lack timely measures. — JEV jev-1.14.0
- ✅ _supported_: Biobot raised a Series A round in 2021. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 836 in / 434 out · 4062 ms · $0.00873</sub>

### 3. DEVIL → CEO (round 1) — _rewritten ×1_

A 2021 Series A shows investors paid, not that cities will. The CEO admits not knowing how many cities pay, which is the core risk. The evidence shows target cities and budget line items, not a funded purchase. Inference: city budgets may treat this as a one-off pilot.

**Assumptions:**

- City budgets may treat wastewater monitoring as a one-off pilot, not recurring spend

**Cites:** `biobot_case_p11_c5` — HBS case 821-045, "Biobot Analytics", p.11

- ✅ _supported_: A 2021 Series A shows investors paid for the company's shares. — A $20M Series A in 2021 with named investors implies equity purchased.
- ✅ _supported_: The evidence shown in the meeting includes target cities and budget line items. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2862 in / 622 out · 7020 ms · $0.01499</sub>

## Topic: Market size (TAM)

### 4. VC → the panel (round 2)

The founders counted 870 potential US and Canadian cities, but that is a count, not dollars. What annual price per city would you charge, and how many of those cities would plausibly buy recurring service? Please show the maths, ideally by two methods, such as top-down budgets versus bottom-up city count.

**Cites:** `biobot_fact_cities_including_smaller` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: The founders counted 870 potential US and Canadian cities. — JEV jev-1.14.0
- ✅ _supported_: The 870-city figure is a count of cities, not a dollar revenue figure. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 1595 in / 320 out · 3906 ms · $0.00925</sub>

### 5. MODERATOR → CFO (round 2)

This question will be taken up by the CFO.

<sub>jev:jev-latest · 482 in / 43 out · 163 ms · $0.00000</sub>

### 6. CFO → VC (round 2)

Estimates range from about $39M to $744M per year; I would defend the bottom-up in a seed deck, because it rests on the 870-city count and the $15k public floor, and the share-of-water-market figure ignores city budget reality. The floor is a minimum, not proven average pricing, and buyer count is unverified.

**Maths** (✅ checks out): `870 cities x 30% buying x $15,000 = $3.9M (see bottom-up estimate)`

**Assumptions:**

- 30% of cities buy recurring service (assumption)
- price near the $15k floor

| Estimate | Maths | Check | Assumptions |
|---|---|---|---|
| Bottom-up city count (units x price) | `870 cities x 30% recurring buyers = 261 cities; 261 x $15,000 = $3.92M per year; at 71 large cities x 30% = 21 cities x $150,000 = $3.2M extra; total about $7.1M` | ❌ does not add up | 30% of cities buy recurring service (assumption); small cities pay the $15k floor; large cities pay $150k (assumption); buyers renew annually |
| Beachhead budgets (top-down) | `$110B x 20% = $22B public health; $22B x 0.1% = $22M; $22M x 50% = $11M per year across four cities` | ✅ | 0.1% of public health budget goes to wastewater monitoring (assumption); half is won by Biobot; budget lines are recurring, not one-off pilots |
| Share of existing market (top-down) | `$138.00B x 53.93% = $74.42B; $74.42B x 1.00% = $744.23M per year` | ✅ | 1% capture of municipal water market is plausible (assumption); wastewater data spend is comparable to water spend |

**Cites:** `biobot_fact_cities_including_smaller` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_public_business_and_market_02` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: Estimates range from about $39M to $744M per year. — JEV jev-1.14.0
- ❔ _unverifiable_: Bottom-up method: 870 cities x 30% recurring buyers = 261 cities. — JEV jev-1.14.0
- ❔ _unverifiable_: Bottom-up method: 261 cities x $15,000 = $3.92M per year. — 261 cities derived from an unsupported 30% rate; arithmetic is correct.
- ❔ _unverifiable_: Bottom-up method: 71 large cities x 30% = 21 cities paying $150,000 each adds $3.2M per year. — 71 cities supported; 30% rate and $150,000 price not in evidence.
- ❔ _unverifiable_: Bottom-up method total is about $7.1M per year. — JEV jev-1.14.0
- ✅ _supported_: Top-down beachhead method: $110B x 20% = $22B public health budget. — JEV jev-1.14.0
- ❔ _unverifiable_: Top-down beachhead method: 0.1% of the $22B public health budget is $22M. — $22B supported; the 0.1% share is not in evidence, though arithmetic is correct.
- ❔ _unverifiable_: Top-down beachhead method: half of $22M, or $11M per year across four cities. — $22B supported; the half-of-$22M assumption is not in evidence.
- ✅ _supported_: The bottom-up estimate rests on an 870-city count and a $15k public price floor. — 870 cities and $15,000 minimum charge both appear in evidence.
- ❔ _unverifiable_: Share-of-market method: $138.00B x 53.93% = $74.42B. — 53.93% supported; the $138B base is not in evidence.
- ❔ _unverifiable_: Share-of-market method: 1.00% of $74.42B = $744.23M per year. — JEV jev-1.14.0
- ✅ _supported_: The $15k public floor is a minimum price, not proven average pricing. — JEV jev-1.14.0
- ❔ _unverifiable_: 870 cities x 30% buying x $15,000 = $3.9M per year. — Inputs 870 and $15k exist; the 30% buying rate is not in evidence.
- 🧮 _miscalculated_: Bottom-up city count (units x price): 870 cities x 30% recurring buyers = 261 cities; 261 x $15,000 = $3.92M per year; at 71 large cities x 30% = 21 cities x $150,000 = $3.2M extra; total about $7.1M — Estimate's calculation does not add up

<sub>claude-sonnet-5-5 · 2170 in / 1510 out · 9422 ms · $0.01550</sub>

### 7. DEVIL → CFO (round 2)

The CFO's headline "$39M to $744M" range does not match its own estimates, which total about $3.9M to $7.1M bottom-up. It also says the bottom-up "rests on" the $15k floor, yet the 30% buy rate and $150k large-city price are invented, with no evidence behind them. The range is unreliable.

**Maths** (✅ checks out): `870 x 30% = 261 cities; 261 x $15,000 = $3.915M; 71 x 30% = 21 cities; 21 x $150,000 = $3.15M; total = $7.07M`

**Assumptions:**

- 30% of cities buy recurring service
- large cities pay $150k

**Cites:** `biobot_assume_municipal_price_floor` — Sizing input - not from the case (assumption or public source)

- ❔ _unverifiable_: The CFO's headline range of $39M to $744M does not match the CFO's own bottom-up estimates, which total about $3.9M to $7.1M. — JEV jev-1.14.0
- ❔ _unverifiable_: 870 cities multiplied by 30% gives 261 cities. — 870 supported; 30% not in evidence. Arithmetic is correct (261).
- ❔ _unverifiable_: 261 cities multiplied by $15,000 gives $3.915M. — $15,000 supported; 261 cities relies on unsupported 30%. Arithmetic is correct.
- ❔ _unverifiable_: 71 large cities multiplied by 30% gives 21 cities. — 71 supported; 30% rate not in evidence. Arithmetic is correct.
- ❔ _unverifiable_: 21 cities multiplied by $150,000 gives $3.15M. — The $150,000 price is not in evidence. Arithmetic is correct.
- ❔ _unverifiable_: The total of the two bottom-up components is $7.07M. — Bottom-up components are not given in evidence. The sum of 3.915M and 3.15M is 7.065M.

<sub>claude-sonnet-5-5 · 2199 in / 655 out · 4786 ms · $0.00714</sub>

### 8. VC → the panel (round 3) — _follow-up_

Your three estimates span roughly $7M to $744M. Which single input drives that gap, and where does each come from? The 30% buy rate and $150k large-city price look like assumptions, not evidence. Which one input would you test first with real city buyers before it goes in a seed deck?

**Cites:** `biobot_sizing_city_price_floor` — Sizing model (facts + assumptions)

- ❔ _unverifiable_: The speaker's three estimates span roughly $7M to $744M. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 3625 in / 335 out · 4712 ms · $0.00894</sub>

### 9. MODERATOR → CFO (round 3)

This follow-up stays with the CFO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 10. CFO → VC (round 3) — _rewritten ×1_

The Devil is right: my $39M-$744M headline did not match my estimates, which total about $3.9M-$7.1M bottom-up, $11M beachhead and $744M share-of-market. The $744M share-of-water-market input drives the gap. I would test the 30% buy rate first with real city buyers.

**Maths** (✅ checks out): `870 cities x 30% = 261 cities; 261 x $15,000 = $3.915M; 71 x 30% = 21 cities; 21 x $150,000 = $3.15M; $3.915M + $3.15M = $7.07M`

**Assumptions:**

- 30% of cities buy recurring service (assumption)
- large cities pay $150k (assumption)
- small cities pay the $15k floor

**Cites:** `biobot_case_p11_c5` — HBS case 821-045, "Biobot Analytics", p.11; `biobot_sizing_share_of_existing_market` — Sizing model (facts + assumptions)

- ❔ _unverifiable_: 71 cities x 30% = 21 cities. — 71 cities supported; 30% share not in evidence. Arithmetic rounds to 21.
- ❔ _unverifiable_: 21 cities x $150,000 = $3.15M. — JEV jev-1.14.0
- ❔ _unverifiable_: $3.915M + $3.15M = $7.07M. — Evidence has no $3.15M component; the sum's inputs are not covered.
- ❔ _unverifiable_: The speaker's bottom-up estimates total about $3.9M-$7.1M. — JEV jev-1.14.0
- ❔ _unverifiable_: The speaker's beachhead estimate is $11M. — Evidence gives no $11M beachhead estimate; only $110B budgets and 20% health share.
- ❔ _unverifiable_: The speaker's share-of-market estimate is $744M. — JEV jev-1.14.0
- ❔ _unverifiable_: 870 cities x 30% = 261 cities. — 870 is supported, but 30% penetration is not in evidence; arithmetic 261 is correct.
- ❔ _unverifiable_: 261 cities x $15,000 = $3.915M. — Arithmetic correct and $15,000 floor exists, but 261-city assumption is not evidenced.

<sub>claude-sonnet-5-5 · 5218 in / 1667 out · 10798 ms · $0.01746</sub>

### 11. DEVIL → CFO (round 3)

The CFO's pick, the 30% buy rate, is not the weakest input. The $15k floor and $150k large-city price both sit under that same rate, and the CFO has no evidence for either. The 71-city count is also a population threshold (over 500,000), not a buyer list. So the $7.1M is stacked guesses, not a testable number.

**Maths** (✅ checks out): `261 small-city buyers x $15,000 = $3.915M; 21 large-city buyers x $150,000 = $3.15M; $150,000 / $15,000 = 10x price gap on 21 cities = 45% of total`

**Assumptions:**

- The $150k large-city price is an assumption with no source in the evidence shown

**Cites:** `biobot_case_p11_c5` — HBS case 821-045, "Biobot Analytics", p.11

- ❔ _unverifiable_: The $15k price floor and the $150k large-city price both rest on the same 30% buy rate assumption. — JEV jev-1.14.0
- ✅ _supported_: The 71-city count is a population threshold (cities over 500,000 people), not a list of buyers. — JEV jev-1.14.0
- ❔ _unverifiable_: 261 small-city buyers multiplied by $15,000 equals $3.915M. — Arithmetic correct, but 261 buyers assumption and pricing as ARR not established.
- ❔ _unverifiable_: 21 large-city buyers multiplied by $150,000 equals $3.15M. — JEV jev-1.14.0
- ❔ _unverifiable_: The $150,000 large-city price is 10 times the $15,000 price, a 10x price gap applied to 21 cities. — JEV jev-1.14.0
- ❔ _unverifiable_: The 10x price gap on 21 cities accounts for 45% of the total estimate. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2694 in / 678 out · 5242 ms · $0.00812</sub>

## Topic: Customers and who pays

### 12. VC → the panel (round 4)

The case says the founders did not yet know who in a city would buy and pay for the data. Who is your first paying customer today, which department's budget line funds the cheque, and why would they sign now rather than treat it as a one-off pilot?

**Cites:** `biobot_fact_buyer_unclear` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_case_p12_c4` — HBS case 821-045, "Biobot Analytics", p.12


<sub>claude-sonnet-5-5 · 3829 in / 291 out · 3549 ms · $0.00930</sub>

### 13. MODERATOR → CEO (round 4)

This question will be taken up by the CEO.

<sub>jev:jev-latest · 469 in / 43 out · 241 ms · $0.00000</sub>

### 14. CEO → VC (round 4) — _rewritten ×1_

I have no paying customer today, and the evidence does not name the budget line. Exhibit 5 lists the mayor and staff as end users, with NYC, Boston, SF and Washington DC as lead targets. The case says the buyer was unclear in 2017, so I would sign one funded pilot first.

**Cites:** `biobot_ex5_municipal_governments` — Exhibit 5 (HBS case 821-045, "Biobot Analytics"); `biobot_fact_buyer_unclear` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: The speaker has no paying customer today. — As of 2017, no customer and unclear buyer; 'today' timing is assumed.
- ✅ _supported_: Exhibit 5 lists the mayor and staff as end users. — JEV jev-1.14.0
- ✅ _supported_: Exhibit 5 lists NYC, Boston, SF and Washington DC as lead targets. — JEV jev-1.14.0
- ✅ _supported_: The case says the buyer was unclear in 2017. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 4764 in / 591 out · 5903 ms · $0.01423</sub>

### 15. DEVIL → CEO (round 4)

The CEO's "fund one pilot first" dodges the VC's question about why anyone signs now. Exhibit 5 names mayor and staff as end users, not buyers, and lists public health departments, EMS and police data as competition. Inference: a pilot may be a one-off that never recurs, and the CEO gives no renewal trigger.

**Cites:** `biobot_ex5_municipal_governments` — Exhibit 5 (HBS case 821-045, "Biobot Analytics"); `biobot_case_p12_c4` — HBS case 821-045, "Biobot Analytics", p.12

- ✅ _supported_: Exhibit 5 names mayor and staff as end users, not buyers. — JEV jev-1.14.0
- ✅ _supported_: Exhibit 5 lists public health departments, EMS and police data as competition. — JEV jev-1.14.0
- ❔ _unverifiable_: The CEO gives no renewal trigger for the pilot. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2592 in / 370 out · 3463 ms · $0.00728</sub>

### 16. VC → the panel (round 5) — _follow-up_

You say you would sign one funded pilot first, but the case notes Ratti's warning that you don't know who will pay for this data (biobot_case_p12_c4). What evidence do you have today of willingness to pay, such as a pilot, budget line or letter of intent, and how long would a city sales cycle take?

**Cites:** `biobot_case_p12_c4` — HBS case 821-045, "Biobot Analytics", p.12


<sub>claude-sonnet-5-5 · 3125 in / 330 out · 3273 ms · $0.00797</sub>

### 17. MODERATOR → CEO (round 5)

This follow-up stays with the CEO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 18. CEO → VC (round 5) — _rewritten ×1_

I have no evidence of willingness to pay: no pilot, budget line or letter of intent is in the evidence I can see. Ratti's warning that we don't know who will pay is fair (p.12). I cannot state a sales cycle; the case only says city procurement has hurdles. I'd test willingness to pay with named buyers first.

**Cites:** `biobot_case_p12_c4` — HBS case 821-045, "Biobot Analytics", p.12; `biobot_fact_buyer_unclear` — Curated fact (HBS case 821-045, "Biobot Analytics")

- ✅ _supported_: Ratti warns that it is not known who will pay (p.12). — JEV jev-1.14.0
- ✅ _supported_: The case states that city procurement has hurdles. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 4392 in / 648 out · 7642 ms · $0.01350</sub>

### 19. DEVIL → CEO (round 5)

The CEO's fix, testing willingness to pay with named buyers, is circular. Exhibit 5 lists NYC, Boston, SF and DC only as lead targets, and its end user is the mayor and staff, not a budget holder. Naming cities without naming a payer repeats the gap Ratti flagged. Inference: this leaves no plan for sales cycle or pricing.

**Cites:** `biobot_ex5_municipal_governments` — Exhibit 5 (HBS case 821-045, "Biobot Analytics"); `biobot_case_p12_c4` — HBS case 821-045, "Biobot Analytics", p.12

- ❌ _contradicted_: Exhibit 5 lists NYC, Boston, SF and DC only as lead targets. — Exhibit 5 lists those cities for municipal governments only; other segments exist, including private sector.
- ✅ _supported_: Exhibit 5's end user is the mayor and staff, not a budget holder. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2051 in / 388 out · 3920 ms · $0.00624</sub>

## Topic: Technology and defensibility

### 20. VC → the panel (round 6)

Topic shift: what stops a competitor, a university lab or a city's own utility from copying Biobot's wastewater opioid measurement? Beyond the MIT spin-out origin, what specific technology, data or access advantage would you still hold if a city ran its own sampling and sent samples to a commercial lab?

- ✅ _supported_: Biobot is a spin-out from MIT. — Founders' MIT research origins and MIT programs; TLO claimed no IP, so 'spin-out' is loose.
- ✅ _supported_: A city could run its own wastewater sampling and send samples to a commercial lab. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2459 in / 386 out · 3646 ms · $0.00659</sub>

### 21. MODERATOR → CTO (round 6)

This question will be taken up by the CTO.

<sub>jev:jev-latest · 477 in / 42 out · 146 ms · $0.00000</sub>

### 22. CTO → VC (round 6)

Candidly, in 2017 the founders believed they could differentiate without proprietary technology (p.12), so lab measurement alone is copyable. What is proprietary: US11714030B2, a sampling patent granted 2023-08-01, covering upstream modular sampling only, not all wastewater surveillance. Dashboards, API and alerts are product features; I cannot verify a data moat.

**Cites:** `biobot_fact_differentiate_on_analytics` — Curated fact (HBS case 821-045, "Biobot Analytics"); `biobot_public_technology_and_intellectual_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: US11714030B2 is a sampling patent granted on 2023-08-01. — JEV jev-1.14.0
- ✅ _supported_: US11714030B2 covers upstream modular sampling only, not all wastewater surveillance. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 1866 in / 432 out · 4747 ms · $0.01100</sub>

### 23. DEVIL → CTO (round 6)

The CTO leans on a patent granted 2023-08-01 as the moat, yet it covers only upstream modular sampling. A city can skip that hardware and ship plant-inlet samples to any lab, which is the very scenario the VC posed. Inference: the patent protects a niche, not the opioid measurement or the customer relationship.

**Assumptions:**

- Municipal plant-based sampling does not need the upstream modular system, so the patent may not be infringed by a competing lab

**Cites:** `biobot_public_technology_and_intellectual_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_technology_and_intellectual_01` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: A patent was granted on 2023-08-01 and the CTO relies on it as the company's moat. — Grant date matches, but nothing says the CTO relies on it as moat.
- ❔ _unverifiable_: The patent covers only upstream modular sampling. — Title says upstream modular sampling; actual claim scope is not given.
- ❔ _unverifiable_: The scenario of a city shipping plant-inlet samples to any lab is the scenario the VC posed in the meeting. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 1762 in / 489 out · 6235 ms · $0.00595</sub>

### 24. VC → the panel (round 7) — _follow-up_

The patent grant on 2023-08-01 covers upstream modular sampling, but municipal work uses plant samples shipped to Biobot. What measurable advantage do you hold today, such as accuracy, turnaround or coverage, and how many months and dollars would a lab need to replicate it? Please separate what you can verify from opinion.

**Cites:** `biobot_public_technology_and_intellectual_03` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_technology_and_intellectual_01` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The patent grant dated 2023-08-01 covers upstream modular sampling. — JEV jev-1.14.0
- ✅ _supported_: Municipal wastewater work currently uses plant samples shipped to Biobot. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2536 in / 397 out · 3753 ms · $0.00715</sub>

### 25. MODERATOR → CTO (round 7)

This follow-up stays with the CTO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 26. CTO → VC (round 7)

Verifiable today: results in 3-5 days after collection, a company-stated best case, not a guarantee (2026-10-22 source). I cannot verify accuracy, coverage or replication cost in months or dollars; the evidence is silent. The Devil is right that the patent covers upstream sampling, not plant-inlet samples, so it is no moat there. Opinion: speed and workflow are copyable.

**Assumptions:**

- Plant-inlet municipal sampling may not use the patented upstream modular system, so the patent may not block a competing lab (Devil's assumption, unverified)

**Cites:** `biobot_public_current_product_catalog_02` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_technology_and_intellectual_03` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: Results are verifiable today to take 3-5 days after sample collection, per a company-stated best case and not a guarantee (2026-10-22 source). — JEV jev-1.14.0
- ✅ _supported_: The company's stated 3-5 day turnaround is a best case, not a guaranteed result time. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2058 in / 608 out · 6157 ms · $0.00683</sub>

### 27. DEVIL → CTO (round 7)

The CTO concedes the patent is no moat, then offers 3-5 days as the verifiable edge, but that figure is a vendor best case, not an audited service level. Inference: with no accuracy, coverage or replication-cost data, a city has no measurable reason to pay Biobot rather than a cheaper lab.

**Assumptions:**

- A competing commercial lab could offer similar turnaround at lower price (inference, unverified)

**Cites:** `biobot_public_company_and_offering_02` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_current_product_catalog_02` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: The 3-5 day turnaround figure is a vendor best case rather than an audited service level. — JEV jev-1.14.0
- ❔ _unverifiable_: No accuracy, coverage or replication-cost data has been provided. — Evidence shows only vendor turnaround claims; absence of accuracy/coverage data across all materials isn't confirmable.

<sub>claude-sonnet-5-5 · 2014 in / 448 out · 4007 ms · $0.00622</sub>

## Topic: Pricing and unit economics

### 28. VC → the panel (round 8)

A local report dated 2025-11-24 says municipalities pay at least $15,000 per year. What is your actual price per site or city, what does it cost you to serve one, collection kits, shipping and lab testing, and what gross margin does that leave? Please separate measured figures from estimates.

**Cites:** `biobot_public_company_and_offering_05` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: A local report dated 2025-11-24 says municipalities pay at least $15,000 per year. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2611 in / 322 out · 3930 ms · $0.00694</sub>

### 29. MODERATOR → CFO (round 8)

This question will be taken up by the CFO.

<sub>jev:jev-latest · 487 in / 43 out · 197 ms · $0.00000</sub>

### 30. CFO → VC (round 8)

Measured: only a reported floor of $15,000 per municipality per year (Cambridge Day, 2025-11-24), not a contract average. I cannot verify our per-site cost for kits, shipping or lab testing; the evidence is silent. The cost and margin below are illustrative estimates, not measured figures.

**Maths** (✅ checks out): `$15,000 price - $6,000 assumed cost to serve = $9,000 gross profit; $9,000 / $15,000 = 60% gross margin`

**Assumptions:**

- Assumed cost to serve one municipality is $6,000 per year (kits, shipping, lab); not in evidence
- One contract at the $15,000 floor covers one city
- Price is the floor; true average unknown

**Cites:** `biobot_public_company_and_offering_05` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ✅ _supported_: A reported floor of $15,000 per municipality per year exists, per Cambridge Day dated 2025-11-24. — JEV jev-1.14.0
- ✅ _supported_: The $15,000 figure is a reported floor, not a measured contract average. — JEV jev-1.14.0
- ❔ _unverifiable_: Gross profit is $9,000 per municipality per year ($15,000 price minus $6,000 assumed cost to serve). — JEV jev-1.14.0
- ❔ _unverifiable_: Gross margin is 60% ($9,000 divided by $15,000). — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2164 in / 663 out · 5496 ms · $0.00714</sub>

### 31. DEVIL → CFO (round 8)

The CFO's 60% gross margin rests on an invented $6,000 cost to serve, set against a price that is only a reported floor. Inference: the margin figure is unfalsifiable, and it ignores any uncosted labor, support or analytics. The evidence also says many sites were pro-bono or federally funded, so realized revenue per site may sit below $15,000.

**Maths** (✅ checks out): `$15,000 - $6,000 = $9,000; $9,000 / $15,000 = 60%; at an assumed $12,000 cost, ($15,000 - $12,000) / $15,000 = 20%`

**Assumptions:**

- Cost to serve could be $12,000 per city (illustrative stress case, not evidence)

**Cites:** `biobot_public_business_and_market_02` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: The CFO's 60% gross margin rests on a $6,000 cost to serve. — JEV jev-1.14.0
- ❔ _unverifiable_: At an assumed $12,000 cost to serve, the gross margin on $15,000 is 20%. — JEV jev-1.14.0
- ✅ _supported_: The price used in the CFO's margin calculation is only a reported floor. — JEV jev-1.14.0
- ✅ _supported_: Many sites were pro-bono or federally funded according to the evidence shown. — JEV jev-1.14.0
- ✅ _supported_: $15,000 minus $6,000 equals $9,000. — Arithmetic is correct: 15,000 - 6,000 = 9,000. The evidence only supplies the $15,000 figure.
- ✅ _supported_: $9,000 divided by $15,000 equals 60%. — 9,000/15,000 = 0.6, so 60% is correct arithmetic.

<sub>claude-sonnet-5-5 · 2242 in / 683 out · 7830 ms · $0.00731</sub>

### 32. VC → the panel (round 9) — _follow-up_

The Devil's stress case moves gross margin from 60% to 20% on one unmeasured cost input. Which price or cost-to-serve assumption breaks your model, and at what number? What annual contract volume covers your fixed costs, and what are your sources for each input? Please separate measured figures from guesses.

**Cites:** `biobot_public_business_and_market_06` — Biobot public data and source ledger (public sources, checked 2026-10-22); `biobot_public_business_and_market_02` — Biobot public data and source ledger (public sources, checked 2026-10-22)

- ❔ _unverifiable_: The Devil's stress case moves gross margin from 60% to 20% on one unmeasured cost input. — JEV jev-1.14.0
- ✅ _supported_: The Devil's stress case's cost input that drives the gross margin move from 60% to 20% is unmeasured. — Gross margin and cost data are not disclosed; no measured cost input exists.

<sub>claude-sonnet-5-5 · 2771 in / 389 out · 5317 ms · $0.00747</sub>

### 33. MODERATOR → CFO (round 9)

This follow-up stays with the CFO, who gave the answer being probed.

<sub>code · 0 in / 0 out · 0 ms · $0.00000</sub>

### 34. CFO → VC (round 9) — _rewritten ×1_

The $15,000 price is only a reported floor (2025-11-24); the $6,000 cost is my invented placeholder, so 60% margin is unproven. Cost to serve is the input that matters most: at $12,000 margin falls to 20%, and it breaks near $15,000. I cannot verify fixed costs or break-even contract volume.

**Maths** (❌ does not add up): `$15,000 - $6,000 = $9,000; $9,000 / $15,000 = 60%; $15,000 - $12,000 = $3,000; $3,000 / $15,000 = 20%; margin reaches 0% at cost = $15,000`

**Assumptions:**

- Cost to serve is $6,000 per city per year (not in evidence)
- Stress-case cost is $12,000 (illustrative)
- Realized price equals the $15,000 floor

**Cites:** `biobot_sizing_city_price_floor` — Sizing model (facts + assumptions)

- ✅ _supported_: The $15,000 price is only a reported floor (dated 2025-11-24). — JEV jev-1.14.0
- ❔ _unverifiable_: $3,000 divided by $15,000 equals a 20% margin. — JEV jev-1.14.0
- ❔ _unverifiable_: Margin reaches 0% when cost equals $15,000. — JEV jev-1.14.0
- ❔ _unverifiable_: At a $12,000 cost to serve, margin falls to 20%. — JEV jev-1.14.0
- ❔ _unverifiable_: Margin breaks even near a $15,000 cost to serve. — JEV jev-1.14.0
- ❔ _unverifiable_: $15,000 minus $6,000 equals $9,000. — No valid supporting citation
- ❔ _unverifiable_: $9,000 divided by $15,000 equals a 60% margin. — JEV jev-1.14.0
- ❔ _unverifiable_: $15,000 minus $12,000 equals $3,000. — JEV jev-1.14.0
- 🧮 _miscalculated_: $15,000 - $6,000 = $9,000; $9,000 / $15,000 = 60%; $15,000 - $12,000 = $3,000; $3,000 / $15,000 = 20%; margin reaches 0% at cost = $15,000 — Shown calculation does not add up

<sub>claude-sonnet-5-5 · 4943 in / 1183 out · 9681 ms · $0.01705</sub>

### 35. DEVIL → CFO (round 9)

The CFO says margin "breaks near $15,000" cost, but that is just where price equals cost, a trivial zero-margin point, not a real failure threshold. Inference: with unknown fixed costs, the model breaks far earlier, since 20% margin on $15,000 is only $3,000 per city. The CFO also never offered a contract volume.

**Maths** (✅ checks out): `$15,000 x 20% = $3,000 gross profit per city per year`

**Assumptions:**

- Cost to serve is $12,000 per city (illustrative stress case, not evidence)
- Realized price equals the $15,000 floor

**Cites:** `biobot_sizing_city_price_floor` — Sizing model (facts + assumptions)

- ❔ _unverifiable_: A 20% margin on $15,000 yields $3,000 gross profit per city per year. — JEV jev-1.14.0

<sub>claude-sonnet-5-5 · 2538 in / 507 out · 4976 ms · $0.00754</sub>

## Fact-checks

| Claim | Speaker | Type | Verdict | Confidence | Checker | Evidence | Note |
|---|---|---|---|---:|---|---|---|
| Biobot's product or service is what the speaker asks the audience to describe in one sentence. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks what Biobot does, for whom, and why now is the right time to build it. | VC | opinion | · not_checked |  |  |  |  |
| Biobot turns wastewater into near-real-time data on community drug consumption. | CEO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_case_p6_c1`, `biobot_ex6_prescription_illicit_drugs`, `biobot_fact_opioids_first` | Evidence shows wastewater drug-consumption tracking, with real-time monitoring aims and alert lead times. |
| Biobot starts with opioids as its focus, serving city governments that currently lack timely measures. | CEO | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_fact_opioids_first`, `biobot_case_p11_c4`, `biobot_ex5_municipal_governments`, `biobot_case_p11_c5` | JEV jev-1.14.0 |
| The opioid crisis is pressing. | CEO | opinion | · not_checked |  |  |  |  |
| Biobot raised a Series A round in 2021. | CEO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_public_funding_and_investors_03`, `biobot_public_funding_and_investors_05` | JEV jev-1.14.0 |
| The 2021 Series A shows investor backing for Biobot. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify from the evidence shown how many cities pay for Biobot's service. | CEO | opinion | · not_checked |  |  |  |  |
| Cities currently lack timely local measures of opioid consumption. | CEO | forecast | · not_checked |  |  |  |  |
| A 2021 Series A shows investors paid for the company's shares. | DEVIL | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_public_funding_and_investors_05` | A $20M Series A in 2021 with named investors implies equity purchased. |
| A 2021 Series A does not show that cities will pay for the product. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CEO admitted in this meeting not knowing how many cities pay. | DEVIL | opinion | · not_checked |  |  |  |  |
| The number of cities that pay for the product is the core risk. | DEVIL | opinion | · not_checked |  |  |  |  |
| The evidence shown in the meeting includes target cities and budget line items. | DEVIL | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_ex7_mayor_s_health_line`, `biobot_ex7_environmental_hazards` | JEV jev-1.14.0 |
| The evidence shown in the meeting does not include a funded purchase by a city. | DEVIL | opinion | · not_checked |  |  |  |  |
| City budgets may treat this as a one-off pilot rather than recurring spend. | DEVIL | forecast | · not_checked |  |  |  |  |
| City budgets may treat wastewater monitoring as a one-off pilot, not recurring spend. | DEVIL | forecast | · not_checked |  |  |  |  |
| The founders counted 870 potential US and Canadian cities. | VC | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_fact_cities_including_smaller`, `biobot_case_p11_c5` | JEV jev-1.14.0 |
| The 870-city figure is a count of cities, not a dollar revenue figure. | VC | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_fact_cities_including_smaller`, `biobot_case_p11_c5` | JEV jev-1.14.0 |
| The question asks what annual price per city would be charged and how many cities would plausibly buy recurring service. | VC | opinion | · not_checked |  |  |  |  |
| The speaker requests a sizing analysis using two methods, such as top-down budgets and bottom-up city count. | VC | opinion | · not_checked |  |  |  |  |
| Estimates range from about $39M to $744M per year. | CFO | number | ❔ unverifiable | 0.98 | jev:jev-latest |  | JEV jev-1.14.0 |
| Bottom-up method: 870 cities x 30% recurring buyers = 261 cities. | CFO | number | ❔ unverifiable | 0.79 | jev:jev-latest |  | JEV jev-1.14.0 |
| Bottom-up method: 261 cities x $15,000 = $3.92M per year. | CFO | number | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_02` | 261 cities derived from an unsupported 30% rate; arithmetic is correct. |
| Bottom-up method: 71 large cities x 30% = 21 cities paying $150,000 each adds $3.2M per year. | CFO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_over_500k` | 71 cities supported; 30% rate and $150,000 price not in evidence. |
| Bottom-up method total is about $7.1M per year. | CFO | number | ❔ unverifiable | 0.97 | jev:jev-latest |  | JEV jev-1.14.0 |
| Small cities pay the $15k floor and large cities pay $150k. | CFO | forecast | · not_checked |  |  |  |  |
| Buyers renew annually. | CFO | forecast | · not_checked |  |  |  |  |
| Top-down beachhead method: $110B x 20% = $22B public health budget. | CFO | number | ✅ supported | 0.89 | jev:jev-latest | `biobot_fact_beachhead_public_health_spend` | JEV jev-1.14.0 |
| Top-down beachhead method: 0.1% of the $22B public health budget is $22M. | CFO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_beachhead_public_health_spend` | $22B supported; the 0.1% share is not in evidence, though arithmetic is correct. |
| Top-down beachhead method: half of $22M, or $11M per year across four cities. | CFO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_beachhead_public_health_spend` | $22B supported; the half-of-$22M assumption is not in evidence. |
| 0.1% of public health budget goes to wastewater monitoring. | CFO | forecast | · not_checked |  |  |  |  |
| The bottom-up estimate rests on an 870-city count and a $15k public price floor. | CFO | fact | ✅ supported | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_including_smaller`, `biobot_public_business_and_market_02` | 870 cities and $15,000 minimum charge both appear in evidence. |
| Biobot wins half of the beachhead spend. | CFO | forecast | · not_checked |  |  |  |  |
| Budget lines are recurring rather than one-off pilots. | CFO | forecast | · not_checked |  |  |  |  |
| Share-of-market method: $138.00B x 53.93% = $74.42B. | CFO | number | ❔ unverifiable | 0.50 | claude-sonnet-5-5 (escalated) | `biobot_fact_municipal_share_of_water_market` | 53.93% supported; the $138B base is not in evidence. |
| Share-of-market method: 1.00% of $74.42B = $744.23M per year. | CFO | number | ❔ unverifiable | 0.94 | jev:jev-latest |  | JEV jev-1.14.0 |
| A 1% capture of the municipal water market is plausible. | CFO | opinion | · not_checked |  |  |  |  |
| Wastewater data spend is comparable to water spend. | CFO | forecast | · not_checked |  |  |  |  |
| The share-of-water-market figure ignores city budget reality. | CFO | opinion | · not_checked |  |  |  |  |
| The bottom-up approach is defensible in a seed deck. | CFO | opinion | · not_checked |  |  |  |  |
| The $15k public floor is a minimum price, not proven average pricing. | CFO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_public_business_and_market_02`, `biobot_public_company_and_offering_05` | JEV jev-1.14.0 |
| The buyer count is unverified. | CFO | opinion | · not_checked |  |  |  |  |
| 870 cities x 30% buying x $15,000 = $3.9M per year. | CFO | number | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_including_smaller`, `biobot_public_business_and_market_02` | Inputs 870 and $15k exist; the 30% buying rate is not in evidence. |
| 30% of cities buy recurring wastewater service. | CFO | forecast | · not_checked |  |  |  |  |
| Cities will pay approximately the $15k floor price. | CFO | forecast | · not_checked |  |  |  |  |
| Bottom-up city count (units x price): 870 cities x 30% recurring buyers = 261 cities; 261 x $15,000 = $3.92M per year; at 71 large cities x 30% = 21 cities x $150,000 = $3.2M extra; total about $7.1M | CFO | number | 🧮 miscalculated | 1.00 | python |  | Estimate's calculation does not add up |
| The CFO's headline range of $39M to $744M does not match the CFO's own bottom-up estimates, which total about $3.9M to $7.1M. | DEVIL | fact | ❔ unverifiable | 0.98 | jev:jev-latest |  | JEV jev-1.14.0 |
| 30% of cities buy the recurring service. | DEVIL | forecast | · not_checked |  |  |  |  |
| Large cities pay $150k. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CFO says the bottom-up estimate rests on a $15k price floor. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 30% city buy rate and the $150k large-city price used in the CFO's estimate have no evidence behind them. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's headline range is unreliable. | DEVIL | opinion | · not_checked |  |  |  |  |
| 870 cities multiplied by 30% gives 261 cities. | DEVIL | number | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_including_smaller` | 870 supported; 30% not in evidence. Arithmetic is correct (261). |
| 261 cities multiplied by $15,000 gives $3.915M. | DEVIL | number | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_02` | $15,000 supported; 261 cities relies on unsupported 30%. Arithmetic is correct. |
| 71 large cities multiplied by 30% gives 21 cities. | DEVIL | number | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_over_500k` | 71 supported; 30% rate not in evidence. Arithmetic is correct. |
| 21 cities multiplied by $150,000 gives $3.15M. | DEVIL | number | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) |  | The $150,000 price is not in evidence. Arithmetic is correct. |
| The total of the two bottom-up components is $7.07M. | DEVIL | number | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) |  | Bottom-up components are not given in evidence. The sum of 3.915M and 3.15M is 7.065M. |
| The speaker's three estimates span roughly $7M to $744M. | VC | number | ❔ unverifiable | 0.89 | jev:jev-latest |  | JEV jev-1.14.0 |
| The speaker asks which single input drives the gap between the three estimates. | VC | opinion | · not_checked |  |  |  |  |
| The 30% buy rate looks like an assumption rather than evidence. | VC | opinion | · not_checked |  |  |  |  |
| The $150k large-city price looks like an assumption rather than evidence. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks which one input should be tested first with real city buyers before it goes into a seed deck. | VC | opinion | · not_checked |  |  |  |  |
| The Devil is right about the speaker's headline figure. | CFO | opinion | · not_checked |  |  |  |  |
| 71 cities x 30% = 21 cities. | CFO | number | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_over_500k` | 71 cities supported; 30% share not in evidence. Arithmetic rounds to 21. |
| 21 cities x $150,000 = $3.15M. | CFO | number | ❔ unverifiable | 0.86 | jev:jev-latest |  | JEV jev-1.14.0 |
| $3.915M + $3.15M = $7.07M. | CFO | number | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_case_p11_c5` | Evidence has no $3.15M component; the sum's inputs are not covered. |
| 30% of cities buy a recurring service. | CFO | forecast | · not_checked |  |  |  |  |
| Large cities pay $150k. | CFO | forecast | · not_checked |  |  |  |  |
| Small cities pay the $15k floor. | CFO | forecast | · not_checked |  |  |  |  |
| The speaker's $39M-$744M headline did not match the speaker's estimates. | CFO | opinion | · not_checked |  |  |  |  |
| The speaker's bottom-up estimates total about $3.9M-$7.1M. | CFO | number | ❔ unverifiable | 0.83 | jev:jev-latest |  | JEV jev-1.14.0 |
| The speaker's beachhead estimate is $11M. | CFO | number | ❔ unverifiable | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_case_p11_c5` | Evidence gives no $11M beachhead estimate; only $110B budgets and 20% health share. |
| The speaker's share-of-market estimate is $744M. | CFO | number | ❔ unverifiable | 0.86 | jev:jev-latest |  | JEV jev-1.14.0 |
| The $744M share-of-water-market input drives the gap between the headline and the estimates. | CFO | opinion | · not_checked |  |  |  |  |
| The speaker would test the 30% buy rate first with real city buyers. | CFO | forecast | · not_checked |  |  |  |  |
| 870 cities x 30% = 261 cities. | CFO | number | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_fact_cities_including_smaller` | 870 is supported, but 30% penetration is not in evidence; arithmetic 261 is correct. |
| 261 cities x $15,000 = $3.915M. | CFO | number | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_02` | Arithmetic correct and $15,000 floor exists, but 261-city assumption is not evidenced. |
| The CFO's 30% buy rate is not the weakest input in the $7.1M estimate. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $150k large-city price is an assumption with no source in the evidence shown. | DEVIL | opinion | · not_checked |  |  |  |  |
| The $15k price floor and the $150k large-city price both rest on the same 30% buy rate assumption. | DEVIL | fact | ❔ unverifiable | 0.98 | jev:jev-latest |  | JEV jev-1.14.0 |
| The CFO has no evidence for the $15k price floor or the $150k large-city price. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 71-city count is a population threshold (cities over 500,000 people), not a list of buyers. | DEVIL | fact | ✅ supported | 0.97 | jev:jev-latest | `biobot_case_p11_c5`, `biobot_fact_cities_over_500k` | JEV jev-1.14.0 |
| The $7.1M figure is built from stacked guesses and is not a testable number. | DEVIL | opinion | · not_checked |  |  |  |  |
| 261 small-city buyers multiplied by $15,000 equals $3.915M. | DEVIL | number | ❔ unverifiable | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_02`, `biobot_case_p11_c5` | Arithmetic correct, but 261 buyers assumption and pricing as ARR not established. |
| 21 large-city buyers multiplied by $150,000 equals $3.15M. | DEVIL | number | ❔ unverifiable | 0.89 | jev:jev-latest |  | JEV jev-1.14.0 |
| The $150,000 large-city price is 10 times the $15,000 price, a 10x price gap applied to 21 cities. | DEVIL | number | ❔ unverifiable | 0.89 | jev:jev-latest |  | JEV jev-1.14.0 |
| The 10x price gap on 21 cities accounts for 45% of the total estimate. | DEVIL | number | ❔ unverifiable | 0.94 | jev:jev-latest |  | JEV jev-1.14.0 |
| The case says the founders did not yet know who in a city would buy and pay for the data. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks who the first paying customer is today. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks which department's budget line funds the cheque. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks why a customer would sign now rather than treat the deal as a one-off pilot. | VC | opinion | · not_checked |  |  |  |  |
| The speaker has no paying customer today. | CEO | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_fact_alm_stay_at_mit`, `biobot_fact_buyer_unclear`, `biobot_fact_ratti_who_pays` | As of 2017, no customer and unclear buyer; 'today' timing is assumed. |
| The evidence does not name the budget line. | CEO | opinion | · not_checked |  |  |  |  |
| Exhibit 5 lists the mayor and staff as end users. | CEO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_ex5_municipal_governments` | JEV jev-1.14.0 |
| Exhibit 5 lists NYC, Boston, SF and Washington DC as lead targets. | CEO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_ex5_municipal_governments`, `biobot_case_p11_c5` | JEV jev-1.14.0 |
| The case says the buyer was unclear in 2017. | CEO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_fact_buyer_unclear`, `biobot_case_p12_c4` | JEV jev-1.14.0 |
| The speaker would sign one funded pilot first. | CEO | forecast | · not_checked |  |  |  |  |
| The CEO's "fund one pilot first" dodges the VC's question about why anyone signs now. | DEVIL | opinion | · not_checked |  |  |  |  |
| Exhibit 5 names mayor and staff as end users, not buyers. | DEVIL | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_ex5_municipal_governments`, `biobot_fact_buyer_unclear` | JEV jev-1.14.0 |
| Exhibit 5 lists public health departments, EMS and police data as competition. | DEVIL | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_ex5_municipal_governments` | JEV jev-1.14.0 |
| A pilot may be a one-off that never recurs. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CEO gives no renewal trigger for the pilot. | DEVIL | fact | ❔ unverifiable | 0.83 | jev:jev-latest |  | JEV jev-1.14.0 |
| The speaker says the other party would sign one funded pilot first. | VC | opinion | · not_checked |  |  |  |  |
| The case notes contain a warning from Ratti that the other party does not know who will pay for this data (biobot_case_p12_c4). | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks what evidence the other party currently has of willingness to pay, such as a pilot, budget line or letter of intent. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks how long a city sales cycle would take. | VC | opinion | · not_checked |  |  |  |  |
| The speaker has no evidence of willingness to pay in the evidence they can see. | CEO | opinion | · not_checked |  |  |  |  |
| No pilot, budget line or letter of intent appears in the evidence visible to the speaker. | CEO | opinion | · not_checked |  |  |  |  |
| Ratti warns that it is not known who will pay (p.12). | CEO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_case_p12_c4`, `biobot_fact_buyer_unclear`, `biobot_fact_ratti_who_pays` | JEV jev-1.14.0 |
| The speaker considers Ratti's warning about unknown payers to be fair. | CEO | opinion | · not_checked |  |  |  |  |
| The speaker cannot state a sales cycle from the case. | CEO | opinion | · not_checked |  |  |  |  |
| The case states that city procurement has hurdles. | CEO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_case_p12_c4`, `biobot_fact_access_and_procurement` | JEV jev-1.14.0 |
| The speaker would test willingness to pay with named buyers first. | CEO | forecast | · not_checked |  |  |  |  |
| The CEO's fix, testing willingness to pay with named buyers, is circular. | DEVIL | opinion | · not_checked |  |  |  |  |
| Exhibit 5 lists NYC, Boston, SF and DC only as lead targets. | DEVIL | fact | ❌ contradicted | 0.80 | claude-sonnet-5-5 (escalated) | `biobot_ex5_municipal_governments`, `biobot_case_p11_c5` | Exhibit 5 lists those cities for municipal governments only; other segments exist, including private sector. |
| Exhibit 5's end user is the mayor and staff, not a budget holder. | DEVIL | fact | ✅ supported | 0.95 | jev:jev-latest | `biobot_ex5_municipal_governments` | JEV jev-1.14.0 |
| Naming cities without naming a payer repeats the gap Ratti flagged. | DEVIL | opinion | · not_checked |  |  |  |  |
| The proposal leaves no plan for sales cycle or pricing. | DEVIL | opinion | · not_checked |  |  |  |  |
| The statement asks what would stop a competitor, a university lab, or a city utility from copying Biobot's wastewater opioid measurement. | VC | opinion | · not_checked |  |  |  |  |
| Biobot is a spin-out from MIT. | VC | fact | ✅ supported | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_history_and_operating_02` | Founders' MIT research origins and MIT programs; TLO claimed no IP, so 'spin-out' is loose. |
| Biobot's wastewater opioid measurement could be copied by a competitor, a university lab, or a city's own utility. | VC | forecast | · not_checked |  |  |  |  |
| A city could run its own wastewater sampling and send samples to a commercial lab. | VC | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_public_technology_and_intellectual_01`, `biobot_public_competitors_alternatives_and_03`, `biobot_case_p4_c5` | JEV jev-1.14.0 |
| Beyond its MIT spin-out origin, Biobot must have a specific technology, data, or access advantage to remain defensible against a city running its own sampling. | VC | opinion | · not_checked |  |  |  |  |
| In 2017 the founders believed they could differentiate without proprietary technology (cited as p.12 of the materials). | CTO | opinion | · not_checked |  |  |  |  |
| Lab measurement alone is copyable. | CTO | opinion | · not_checked |  |  |  |  |
| US11714030B2 is a sampling patent granted on 2023-08-01. | CTO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_public_technology_and_intellectual_03` | JEV jev-1.14.0 |
| US11714030B2 covers upstream modular sampling only, not all wastewater surveillance. | CTO | fact | ✅ supported | 0.89 | jev:jev-latest | `biobot_public_technology_and_intellectual_03`, `biobot_case_p5_c5` | JEV jev-1.14.0 |
| Dashboards, API and alerts are product features. | CTO | opinion | · not_checked |  |  |  |  |
| The speaker cannot verify a data moat from the evidence shown. | CTO | opinion | · not_checked |  |  |  |  |
| A patent was granted on 2023-08-01 and the CTO relies on it as the company's moat. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_technology_and_intellectual_03` | Grant date matches, but nothing says the CTO relies on it as moat. |
| The patent covers only upstream modular sampling. | DEVIL | fact | ❔ unverifiable | 0.55 | claude-sonnet-5-5 (escalated) | `biobot_public_technology_and_intellectual_03` | Title says upstream modular sampling; actual claim scope is not given. |
| A city can skip the upstream modular sampling hardware and ship plant-inlet samples to any laboratory. | DEVIL | opinion | · not_checked |  |  |  |  |
| The scenario of a city shipping plant-inlet samples to any lab is the scenario the VC posed in the meeting. | DEVIL | fact | ❔ unverifiable | 0.91 | jev:jev-latest |  | JEV jev-1.14.0 |
| The patent protects a niche rather than the opioid measurement or the customer relationship. | DEVIL | opinion | · not_checked |  |  |  |  |
| Municipal plant-based sampling does not need the upstream modular system, so the patent may not be infringed by a competing lab. | DEVIL | forecast | · not_checked |  |  |  |  |
| The patent grant dated 2023-08-01 covers upstream modular sampling. | VC | fact | ✅ supported | 0.94 | jev:jev-latest | `biobot_public_technology_and_intellectual_03` | JEV jev-1.14.0 |
| Municipal wastewater work currently uses plant samples shipped to Biobot. | VC | fact | ✅ supported | 0.85 | jev:jev-latest | `biobot_public_technology_and_intellectual_01`, `biobot_public_current_product_catalog_02`, `biobot_public_public_measurements_and_07` | JEV jev-1.14.0 |
| The speaker asks what measurable advantage the company holds today, such as accuracy, turnaround or coverage. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks how many months and dollars a lab would need to replicate the claimed advantage. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks for a separation of what can be verified from opinion. | VC | opinion | · not_checked |  |  |  |  |
| Results are verifiable today to take 3-5 days after sample collection, per a company-stated best case and not a guarantee (2026-10-22 source). | CTO | fact | ✅ supported | 0.83 | jev:jev-latest | `biobot_public_current_product_catalog_02`, `biobot_public_company_and_offering_02` | JEV jev-1.14.0 |
| The company's stated 3-5 day turnaround is a best case, not a guaranteed result time. | CTO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_public_current_product_catalog_02`, `biobot_public_company_and_offering_02` | JEV jev-1.14.0 |
| The speaker cannot verify the accuracy, coverage, or replication cost of the system in months or dollars, because the evidence is silent on these. | CTO | opinion | · not_checked |  |  |  |  |
| The Devil stated that the patent covers upstream sampling and not plant-inlet samples. | CTO | opinion | · not_checked |  |  |  |  |
| The speaker agrees that the patent covers upstream sampling and not plant-inlet samples, so it provides no moat for plant-inlet sampling. | CTO | opinion | · not_checked |  |  |  |  |
| Speed and workflow are copyable by competitors. | CTO | opinion | · not_checked |  |  |  |  |
| Plant-inlet municipal sampling may not use the patented upstream modular system, so the patent may not block a competing lab. | CTO | forecast | · not_checked |  |  |  |  |
| The CTO conceded in the meeting that the patent is no moat. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CTO offered 3-5 days as the verifiable edge in the meeting. | DEVIL | opinion | · not_checked |  |  |  |  |
| The 3-5 day turnaround figure is a vendor best case rather than an audited service level. | DEVIL | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_public_company_and_offering_02`, `biobot_public_current_product_catalog_02` | JEV jev-1.14.0 |
| No accuracy, coverage or replication-cost data has been provided. | DEVIL | fact | ❔ unverifiable | 0.60 | claude-sonnet-5-5 (escalated) | `biobot_public_company_and_offering_02`, `biobot_public_named_customer_examples_07` | Evidence shows only vendor turnaround claims; absence of accuracy/coverage data across all materials isn't confirmable. |
| A city has no measurable reason to pay Biobot rather than a cheaper lab, given the absence of accuracy, coverage and replication-cost data. | DEVIL | opinion | · not_checked |  |  |  |  |
| A competing commercial lab could offer similar turnaround at a lower price. | DEVIL | forecast | · not_checked |  |  |  |  |
| A local report dated 2025-11-24 says municipalities pay at least $15,000 per year. | VC | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_public_company_and_offering_05`, `biobot_public_business_and_market_02` | JEV jev-1.14.0 |
| The question asks the speaker for their actual price per site or city. | VC | opinion | · not_checked |  |  |  |  |
| The question asks the speaker for the cost to serve one site or city, including collection kits, shipping and lab testing. | VC | opinion | · not_checked |  |  |  |  |
| The question asks the speaker for the resulting gross margin and to separate measured figures from estimates. | VC | opinion | · not_checked |  |  |  |  |
| A reported floor of $15,000 per municipality per year exists, per Cambridge Day dated 2025-11-24. | CFO | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_public_company_and_offering_05`, `biobot_public_business_and_market_02` | JEV jev-1.14.0 |
| The $15,000 figure is a reported floor, not a measured contract average. | CFO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_public_company_and_offering_05`, `biobot_public_business_and_market_02` | JEV jev-1.14.0 |
| The speaker cannot verify the per-site cost for kits, shipping or lab testing from the evidence shown. | CFO | opinion | · not_checked |  |  |  |  |
| The cost and margin figures in this statement are illustrative estimates, not measured figures. | CFO | opinion | · not_checked |  |  |  |  |
| Gross profit is $9,000 per municipality per year ($15,000 price minus $6,000 assumed cost to serve). | CFO | number | ❔ unverifiable | 0.91 | jev:jev-latest |  | JEV jev-1.14.0 |
| Gross margin is 60% ($9,000 divided by $15,000). | CFO | number | ❔ unverifiable | 0.94 | jev:jev-latest |  | JEV jev-1.14.0 |
| The assumed cost to serve one municipality is $6,000 per year covering kits, shipping and lab testing, which is not in evidence. | CFO | forecast | · not_checked |  |  |  |  |
| One contract at the $15,000 floor covers one city. | CFO | forecast | · not_checked |  |  |  |  |
| The price of $15,000 is the floor and the true average price is unknown. | CFO | opinion | · not_checked |  |  |  |  |
| The CFO's 60% gross margin rests on a $6,000 cost to serve. | DEVIL | fact | ❔ unverifiable | 0.97 | jev:jev-latest |  | JEV jev-1.14.0 |
| At an assumed $12,000 cost to serve, the gross margin on $15,000 is 20%. | DEVIL | number | ❔ unverifiable | 0.82 | jev:jev-latest |  | JEV jev-1.14.0 |
| Cost to serve could be $12,000 per city. | DEVIL | forecast | · not_checked |  |  |  |  |
| The $6,000 cost to serve used in the CFO's 60% gross margin is invented rather than drawn from evidence. | DEVIL | opinion | · not_checked |  |  |  |  |
| The price used in the CFO's margin calculation is only a reported floor. | DEVIL | fact | ✅ supported | 0.79 | jev:jev-latest | `biobot_public_business_and_market_02`, `biobot_public_company_and_offering_05` | JEV jev-1.14.0 |
| The CFO's 60% gross margin figure is unfalsifiable. | DEVIL | opinion | · not_checked |  |  |  |  |
| The CFO's 60% gross margin ignores any uncosted labor, support or analytics. | DEVIL | opinion | · not_checked |  |  |  |  |
| Many sites were pro-bono or federally funded according to the evidence shown. | DEVIL | fact | ✅ supported | 0.98 | jev:jev-latest | `biobot_public_business_and_market_02`, `biobot_public_check_of_the_04` | JEV jev-1.14.0 |
| Realized revenue per site may sit below $15,000. | DEVIL | forecast | · not_checked |  |  |  |  |
| $15,000 minus $6,000 equals $9,000. | DEVIL | number | ✅ supported | 0.97 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_02` | Arithmetic is correct: 15,000 - 6,000 = 9,000. The evidence only supplies the $15,000 figure. |
| $9,000 divided by $15,000 equals 60%. | DEVIL | number | ✅ supported | 0.97 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_02` | 9,000/15,000 = 0.6, so 60% is correct arithmetic. |
| The Devil's stress case moves gross margin from 60% to 20% on one unmeasured cost input. | VC | fact | ❔ unverifiable | 0.98 | jev:jev-latest |  | JEV jev-1.14.0 |
| The Devil's stress case's cost input that drives the gross margin move from 60% to 20% is unmeasured. | VC | fact | ✅ supported | 0.70 | claude-sonnet-5-5 (escalated) | `biobot_public_business_and_market_06` | Gross margin and cost data are not disclosed; no measured cost input exists. |
| The speaker asks which price or cost-to-serve assumption breaks the model and at what number. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks what annual contract volume covers fixed costs. | VC | opinion | · not_checked |  |  |  |  |
| The speaker asks for sources for each input and asks that measured figures be separated from guesses. | VC | opinion | · not_checked |  |  |  |  |
| The $15,000 price is only a reported floor (dated 2025-11-24). | CFO | fact | ✅ supported | 1.00 | jev:jev-latest | `biobot_public_company_and_offering_05`, `biobot_public_business_and_market_02` | JEV jev-1.14.0 |
| $3,000 divided by $15,000 equals a 20% margin. | CFO | number | ❔ unverifiable | 0.92 | jev:jev-latest |  | JEV jev-1.14.0 |
| Margin reaches 0% when cost equals $15,000. | CFO | number | ❔ unverifiable | 0.97 | jev:jev-latest |  | JEV jev-1.14.0 |
| Cost to serve is assumed to be $6,000 per city per year, which is not in evidence. | CFO | forecast | · not_checked |  |  |  |  |
| Stress-case cost to serve is assumed to be $12,000, as an illustrative figure. | CFO | forecast | · not_checked |  |  |  |  |
| Realized price is assumed to equal the $15,000 floor. | CFO | forecast | · not_checked |  |  |  |  |
| The $6,000 cost figure is an invented placeholder by the speaker, not evidence-based, so a 60% margin is unproven. | CFO | opinion | · not_checked |  |  |  |  |
| Cost to serve is the input that matters most for margin. | CFO | opinion | · not_checked |  |  |  |  |
| At a $12,000 cost to serve, margin falls to 20%. | CFO | number | ❔ unverifiable | 1.00 | jev:jev-latest |  | JEV jev-1.14.0 |
| Margin breaks even near a $15,000 cost to serve. | CFO | number | ❔ unverifiable | 0.97 | jev:jev-latest |  | JEV jev-1.14.0 |
| The speaker cannot verify fixed costs or break-even contract volume from the evidence shown. | CFO | opinion | · not_checked |  |  |  |  |
| $15,000 minus $6,000 equals $9,000. | CFO | number | ❔ unverifiable | 0.00 | claude-sonnet-5-5 (escalated) |  | No valid supporting citation |
| $9,000 divided by $15,000 equals a 60% margin. | CFO | number | ❔ unverifiable | 0.94 | jev:jev-latest |  | JEV jev-1.14.0 |
| $15,000 minus $12,000 equals $3,000. | CFO | number | ❔ unverifiable | 0.79 | jev:jev-latest |  | JEV jev-1.14.0 |
| $15,000 - $6,000 = $9,000; $9,000 / $15,000 = 60%; $15,000 - $12,000 = $3,000; $3,000 / $15,000 = 20%; margin reaches 0% at cost = $15,000 | CFO | number | 🧮 miscalculated | 1.00 | python |  | Shown calculation does not add up |
| The CFO said the margin 'breaks near $15,000' cost. | DEVIL | opinion | · not_checked |  |  |  |  |
| The break point of $15,000 is where price equals cost, a zero-margin point rather than a real failure threshold. | DEVIL | opinion | · not_checked |  |  |  |  |
| With unknown fixed costs, the model breaks far earlier than $15,000. | DEVIL | forecast | · not_checked |  |  |  |  |
| The CFO did not offer a contract volume during the meeting. | DEVIL | opinion | · not_checked |  |  |  |  |
| A 20% margin on $15,000 yields $3,000 gross profit per city per year. | DEVIL | number | ❔ unverifiable | 0.97 | jev:jev-latest |  | JEV jev-1.14.0 |
| The cost to serve each city is $12,000 in an illustrative stress case that is not evidence. | DEVIL | forecast | · not_checked |  |  |  |  |
| The realized price equals the $15,000 floor. | DEVIL | forecast | · not_checked |  |  |  |  |

## VC decision

**pass** — conviction 68%

| Rubric | Score | Why |
|---|---:|---|
| Team | 3/5 | The founders have MIT research origins, though the checker calls 'spin-out' loose because MIT's tech licensing office claimed no IP. They raised a 2021 Series A. Management was candid and conceded gaps on willingness to pay, cost and moat. There is no verified evidence on operating or sales track record. CFO sizing errors (headline range mismatch, miscalculated estimates) lower confidence in financial rigor. |
| Market | 3/5 | There are 870 potential US and Canadian cities, and 71 have over 500,000 people. Beachhead public health budgets total $22B. These are counts and budgets, not buyers. Exhibit 5 lists public health departments, EMS and police data as competing sources. All revenue sizing ($3.9M-$744M) rests on unverified buy rates and prices, so market size is real but its addressable share is unproven. |
| Product | 3/5 | The product turns wastewater into drug-consumption data, starting with opioids. Municipal samples are shipped to Biobot. Results arrive 3-5 days after collection, but that is a company-stated best case, not an audited service level. Accuracy and coverage data are absent. |
| Traction | 2/5 | A 2025 report says municipalities pay at least $15,000 per year, so some paid usage exists. However, many sites were pro-bono or federally funded. The case says the buyer was unclear and that there was no paying customer as of 2017, and Ratti warned no one knows who will pay. No contract count, ARR or renewal data was presented. |
| Moat | 2/5 | Patent US11714030B2, granted 2023-08-01, covers upstream modular sampling only, not all wastewater surveillance. Cities can run their own sampling and send samples to commercial labs. No data moat, accuracy edge or replication cost was evidenced. |
| Timing | 3/5 | The opioid-crisis urgency is opinion rather than verified. Investor appetite is shown by the 2021 Series A. Reliance on federal funding for many sites suggests demand may track grant cycles, and city procurement has hurdles. |
| Business Model | 2/5 | The only measured price is a $15,000-per-year floor, not an average. Cost to serve and gross margin are undisclosed, so the 60% margin rests on an invented $6,000 cost. A $12,000 cost cuts margin to 20%. Fixed costs and break-even volume are unknown. The buyer is end-user mayor's staff, not a named budget holder. |

**For:**

- The working product turns wastewater into community drug-consumption data, and municipal sample-shipping workflows are in place.
- Some municipalities now pay at least $15,000 per year, which is evidence that a price point exists.
- The large reachable base includes 870 US and Canadian cities, 71 with over 500,000 people, and $22B in beachhead public health budgets.
- The company has institutional backing (2021 Series A) and a granted sampling patent.

**Against:**

- No verified paying-customer count, ARR, renewal rate or named budget holder exists. Many sites were pro-bono or federally funded, which raises one-off and grant-dependence risk.
- The moat is weak: the patent covers only upstream modular sampling, and cities can self-sample and use commercial labs.
- Unit economics are unknown. The price is only a floor, cost to serve is undisclosed, and margin could be 20% or lower under plausible costs.
- Market sizing is unreliable: CFO estimates span $3.9M to $744M on invented buy rates and prices, and contain calculation errors.
- City procurement hurdles and an unclear buyer imply long, uncertain sales cycles.

**Terms:** Pass at this time. I would revisit as a small participation (e.g., $1-2M within a larger round with a lead) with tranching tied to verified metrics. Tranche 1 would require at least 10 recurring municipal contracts funded from city budget lines rather than federal grants. Tranche 2 would require more than 80% renewal and demonstrated gross margin above 50% from audited cost-to-serve data.

**Would change my mind:**

- A data-room export of signed municipal contracts showing average price, the funding source (city budget vs. federal grant) and renewal history.
- Measured cost to serve per site (kits, shipping, lab, analyst labor) supporting gross margin of at least 50% at realized prices.
- Evidence of a defensible edge: independent accuracy validation, a proprietary longitudinal dataset, or switching costs that a commercial lab cannot replicate within 12 months.
- Named budget-holding departments with documented sales-cycle length and at least 3 letters of intent converting pilots to multi-year contracts.

**Checked claims the decision relies on:**

- `5318f7e0717c-t2-c1` Biobot turns wastewater into near-real-time data on community drug consumption.
- `5318f7e0717c-t2-c2` Biobot starts with opioids as its focus, serving city governments that currently lack timely measures.
- `5318f7e0717c-t2-c4` Biobot raised a Series A round in 2021.
- `5318f7e0717c-t3-c1` A 2021 Series A shows investors paid for the company's shares.
- `5318f7e0717c-t3-c5` The evidence shown in the meeting includes target cities and budget line items.
- `5318f7e0717c-t4-c1` The founders counted 870 potential US and Canadian cities.
- `5318f7e0717c-t4-c2` The 870-city figure is a count of cities, not a dollar revenue figure.
- `5318f7e0717c-t6-c16` Top-down beachhead method: $110B x 20% = $22B public health budget.
- `5318f7e0717c-t11-c4` The 71-city count is a population threshold (cities over 500,000 people), not a list of buyers.
- `5318f7e0717c-t14-c1` The speaker has no paying customer today.
- `5318f7e0717c-t14-c5` The case says the buyer was unclear in 2017.
- `5318f7e0717c-t15-c2` Exhibit 5 names mayor and staff as end users, not buyers.
- `5318f7e0717c-t15-c3` Exhibit 5 lists public health departments, EMS and police data as competition.
- `5318f7e0717c-t18-c3` Ratti warns that it is not known who will pay (p.12).
- `5318f7e0717c-t18-c6` The case states that city procurement has hurdles.
- `5318f7e0717c-t19-c3` Exhibit 5's end user is the mayor and staff, not a budget holder.
- `5318f7e0717c-t20-c2` Biobot is a spin-out from MIT.
- `5318f7e0717c-t20-c4` A city could run its own wastewater sampling and send samples to a commercial lab.
- `5318f7e0717c-t22-c3` US11714030B2 is a sampling patent granted on 2023-08-01.
- `5318f7e0717c-t22-c4` US11714030B2 covers upstream modular sampling only, not all wastewater surveillance.
- `5318f7e0717c-t24-c1` The patent grant dated 2023-08-01 covers upstream modular sampling.
- `5318f7e0717c-t24-c2` Municipal wastewater work currently uses plant samples shipped to Biobot.
- `5318f7e0717c-t26-c1` Results are verifiable today to take 3-5 days after sample collection, per a company-stated best case and not a guarantee (2026-10-22 source).
- `5318f7e0717c-t26-c2` The company's stated 3-5 day turnaround is a best case, not a guaranteed result time.
- `5318f7e0717c-t27-c3` The 3-5 day turnaround figure is a vendor best case rather than an audited service level.
- `5318f7e0717c-t28-c1` A local report dated 2025-11-24 says municipalities pay at least $15,000 per year.
- `5318f7e0717c-t30-c1` A reported floor of $15,000 per municipality per year exists, per Cambridge Day dated 2025-11-24.
- `5318f7e0717c-t30-c2` The $15,000 figure is a reported floor, not a measured contract average.
- `5318f7e0717c-t31-c3` The price used in the CFO's margin calculation is only a reported floor.
- `5318f7e0717c-t31-c6` Many sites were pro-bono or federally funded according to the evidence shown.
- `5318f7e0717c-t32-c2` The Devil's stress case's cost input that drives the gross margin move from 60% to 20% is unmeasured.

## Cost and model calls

| Role | Tier | Model | Calls | Input | Output | Cache read | Latency (s) | Cost |
|---|---|---|---:|---:|---:|---:|---:|---:|
| vc_decision | decision | claude-opus-5-5 | 1 | 35,345 | 3,465 | 0 | 32.5 | $0.22012 |
| fact_check_hard | check_hard | claude-sonnet-5-5 | 9 | 22,577 | 2,822 | 7,096 | 25.6 | $0.07701 |
| devil | debate | claude-sonnet-5-5 | 10 | 19,319 | 2,068 | 20,637 | 28.3 | $0.06918 |
| vc | debate | claude-sonnet-5-5 | 8 | 21,721 | 1,191 | 13,174 | 18.6 | $0.06269 |
| cfo | debate | claude-sonnet-5-5 | 6 | 13,004 | 2,152 | 11,430 | 22.0 | $0.05553 |
| ceo | debate | claude-sonnet-5-5 | 5 | 9,635 | 962 | 8,676 | 12.8 | $0.03605 |
| cto | debate | claude-sonnet-5-5 | 2 | 3,586 | 448 | 2,160 | 5.6 | $0.01748 |
| vc_opening | debate | claude-sonnet-5-5 | 1 | 867 | 72 | 0 | 2.3 | $0.00677 |
| claim_extract | fast | claude-haiku-5-5 | 27 | 4,699 | 8,619 | 19,864 | 59.7 | $0.00507 |
| fact_check | verdict | jev:jev-latest | 9 | 94,021 | 9,736 | 0 | 15.0 | $0.00000 |
| route | route | jev:jev-latest | 4 | 1,915 | 171 | 0 | 0.7 | $0.00000 |

## Audit

- Retrievals: 114 (27 for answers, 87 for fact-checks)
- Turns with citations that were not retrieved: 0
- Cross-agent memory access attempts: 0

### Events

- `2026-10-22T16:56:27+00:00` **route_decision** {"executive": "cfo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.83, "question_seq": 4}
- `2026-10-22T16:57:23+00:00` **route_decision** {"executive": "ceo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 0.97, "question_seq": 12}
- `2026-10-22T16:57:57+00:00` **route_decision** {"executive": "cto", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 1.0, "question_seq": 20}
- `2026-10-22T16:58:33+00:00` **route_decision** {"executive": "cfo", "model": "jev:jev-latest", "reason": "JEV jev-1.14.0", "confidence": 1.0, "question_seq": 28}
